# Lab 0 — Does the number hold up?

**MA553 · Session 1**

Name:

---

This lab has two parts.

**Part A** shows you how a classification result is produced, one step at a
time. If you have done this before it will be quick. If you have not, it is
all you need for the rest of the lab.

**Part B** takes a result that was produced this way, and asks whether the
number can be trusted.

Work through the cells in order. Do not skip ahead — part of the value comes
from writing down a prediction before you see the answer.

# Part A — How the result was made

## The task

We have records for 569 patients. For each patient there are 30
measurements taken from an image of a tissue sample, and a label saying
whether that sample was **malignant** or **benign**.

The task is to take the 30 measurements for a patient we have not seen, and
decide which of the two labels applies.

Some words we will use:

- A **measurement** is one number describing a patient, such as the average
  radius of the cells. There are 30 of them per patient. Another common word
  for these is *features*.
- The **label** is the answer we are trying to produce: malignant or benign.
- A **model** is a procedure that takes the 30 measurements and returns a
  label. Another word for it is a *classifier*.
- **Training** a model means showing it patients whose labels we already
  know, so that it can work out how to label new ones.

## Step A1 — Load the data and look at it

Run this cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = data.data       # the measurements: one row per patient, 30 columns
y = data.target     # the labels:       1 = benign, 0 = malignant

print(f"Patients:      {X.shape[0]}")
print(f"Measurements:  {X.shape[1]} per patient")
print(f"Benign:        {(y == 1).sum()}")
print(f"Malignant:     {(y == 0).sum()}")

In [ ]:
# The first three patients, showing the first five of their 30 measurements.
pd.DataFrame(X[:3, :5], columns=data.feature_names[:5])

## Step A2 — Split the patients into two groups

We cannot judge a model by asking it about patients it was trained on. It
may simply have memorised them. So we put some patients aside before
training, and use only those to score it afterwards.

- The **training set** is the group the model learns from. Here, 80% of the
  patients.
- The **test set** is the group held back for scoring. Here, the other 20%.

Which patients go into which group is decided at random. The `random_state`
number below fixes that random choice, so that the same number always
produces the same split. It is often called a **seed**. It affects nothing
else: not the data, not the models, not how they are trained.

Run this cell.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 10

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # hold back 20% of the patients
    random_state=SEED,    # fixes which patients are held back
    stratify=y,           # keep the same benign/malignant balance in both groups
)

print(f"Training set: {len(y_train)} patients")
print(f"Test set:     {len(y_test)} patients")

## Step A3 — Train the first model

The first model is **logistic regression**. For today you do not need to
know how it works inside. Treat it as a box: you give it the training
patients, and afterwards it will label new ones.

Two details in the code:

- `StandardScaler` puts all 30 measurements on a comparable scale, because
  some are much larger numbers than others. This is preparation of the data,
  not part of the model.
- `C=0.01` is a **setting** — a number that a person chooses before training,
  which changes how the model behaves. Models usually have several. The
  usual name for these is *hyperparameters*. Remember that this one was set
  to 0.01 by somebody; it will matter later.

`.fit()` is the command that trains a model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

baseline = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.01, max_iter=200),
).fit(X_train, y_train)

print("Trained.")

## Step A4 — Use it, and score it

`.predict()` asks the trained model to label patients. Below we label the
test patients — the ones it has never seen — and compare its answers with
the truth.

In [ ]:
predictions = baseline.predict(X_test)

comparison = pd.DataFrame({
    "model said": predictions[:10],
    "truth":      y_test[:10],
})
comparison["correct?"] = comparison["model said"] == comparison["truth"]
comparison

**Accuracy** is the share of test patients the model labelled correctly.
Nothing more than that. Here it is, computed by hand and then by the
library, to show they are the same thing.

In [ ]:
correct = (predictions == y_test).sum()
total = len(y_test)

print(f"Correct by hand: {correct} out of {total} = {100 * correct / total:.1f}%")
print(f"From the library: {100 * baseline.score(X_test, y_test):.1f}%")

## Step A5 — Train a second model, and compare

The second model is **gradient boosting**. Again, you do not need to know
how it works today. Note one thing though: it is created with no settings
chosen at all, so it uses whatever the library's defaults are.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

proposed = GradientBoostingClassifier(random_state=SEED).fit(X_train, y_train)

baseline_accuracy = baseline.score(X_test, y_test) * 100
proposed_accuracy = proposed.score(X_test, y_test) * 100
difference = proposed_accuracy - baseline_accuracy

print(f"Logistic regression: {baseline_accuracy:.1f}%")
print(f"Gradient boosting:   {proposed_accuracy:.1f}%")
print(f"Difference:          {difference:+.1f} percentage points")

A note on wording. A **percentage point** is the plain difference between
two percentages. Going from 92.1% to 95.6% is a rise of 3.5 percentage
points. It is not a rise of 3.5% — that would mean something else again.
Reports often blur the two, so be careful when you write your own.

## That is the whole workflow

Load the data. Hold some patients back. Train on the rest. Score on the
held-back patients. Compare two models.

`result_card.md` in this folder reports exactly the number you just
produced, and claims that gradient boosting is the better method.

**Before you go on, answer this.**

Look back at Steps A3 and A5. One of the two models was given a setting
chosen by a person; the other was left at its defaults.

1. Which line of code sets that value, and what is it?
2. Who chose it, and how would you know whether it was a good choice?

*Your answer:*

# Part B — Does the number hold up?

You have reproduced the reported result. Now find out what it is worth.

## Step B1 — Predict, before you run anything

In Step A2, `random_state=10` decided which patients were held back. A
different number would hold back a different 114 patients. Nothing else
about the two models would change.

Suppose we repeat the whole thing with 20 different values.

**Write your predictions below, and do not change them afterwards.** You are
marked on your reasoning, never on whether you guessed correctly.

**My predictions.**

1. The difference will range roughly from ____ to ____ percentage points.
2. Out of 20 splits, gradient boosting will win about ____ times.

**My reasoning:**

*Your answer:*

## Step B2 — Repeat it across 20 splits

The function below does exactly what you did in Part A, packed into one
place so we can run it many times. The only thing that changes between runs
is the seed.

Ignore the `tuned_baseline` option for now; it is used in Step B4.

In [ ]:
from sklearn.model_selection import GridSearchCV


def compare(seed, tuned_baseline=False):
    """Run the Part A workflow once, with the given seed.
    Returns how much better gradient boosting did, in percentage points.
    A negative number means logistic regression won."""

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    if tuned_baseline:
        # Logistic regression is allowed to try five values of C,
        # and keeps the best one. Only the training patients are used
        # to make that choice.
        baseline = GridSearchCV(
            make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
            {"logisticregression__C": [0.01, 0.1, 1, 10, 100]},
            cv=5,
        ).fit(X_train, y_train)
    else:
        # Logistic regression as the result card ran it: C fixed at 0.01.
        baseline = make_pipeline(
            StandardScaler(), LogisticRegression(C=0.01, max_iter=200)
        ).fit(X_train, y_train)

    proposed = GradientBoostingClassifier(random_state=seed).fit(X_train, y_train)

    return (proposed.score(X_test, y_test) - baseline.score(X_test, y_test)) * 100

In [ ]:
seeds = range(20)
differences = np.array([compare(s) for s in seeds])

print("Difference for each split, in percentage points:")
print(np.round(differences, 1))
print()
print(f"Average:              {differences.mean():+.1f}")
print(f"Typical spread:       {differences.std():.1f}")
print(f"Smallest to largest:  {differences.min():+.1f} to {differences.max():+.1f}")
print(f"Gradient boosting won {(differences > 0).sum()} of {len(differences)} splits")

In [ ]:
plt.figure(figsize=(8, 3.2))
plt.axhline(0, color="grey", linewidth=1)
plt.plot(list(seeds), differences, "o", color="#8C1D2C")
plt.axhline(differences.mean(), color="#8C1D2C", linestyle="--", linewidth=1,
            label=f"average {differences.mean():+.1f}")
plt.xlabel("seed")
plt.ylabel("difference (percentage points)")
plt.title("Same data, same models, different splits")
plt.legend()
plt.tight_layout()
plt.show()

**Answer here.**

1. Where does the reported result sit among the other 19?
2. How far off was your prediction, and why do you think that is?

*Your answer:*

## Step B3 — What one repeat could and could not have shown

Suppose you had changed the seed only once, and seen a different number.

**Answer here.** What would that single extra run have told you, and what
would it not have told you?

*Your answer:*

## Step B4 — Give both sides the same effort

Go back to your answer at the end of Part A.

Gradient boosting ran with its defaults. Logistic regression ran with `C`
fixed at 0.01, a value somebody picked. Nothing tells us that 0.01 is a good
choice for this data.

**Tuning** a model means trying several values of a setting and keeping the
one that works best. `compare(seed, tuned_baseline=True)` lets logistic
regression try five values of `C` and keep the best, choosing with the
training patients only, so the test patients stay untouched.

Everything else stays exactly as it was.

In [ ]:
differences_tuned = np.array([compare(s, tuned_baseline=True) for s in seeds])

print("Difference for each split, with logistic regression tuned:")
print(np.round(differences_tuned, 1))
print()
print(f"Average:              {differences_tuned.mean():+.1f}")
print(f"Typical spread:       {differences_tuned.std():.1f}")
print(f"Smallest to largest:  {differences_tuned.min():+.1f} to {differences_tuned.max():+.1f}")
print(f"Gradient boosting won {(differences_tuned > 0).sum()} of {len(differences_tuned)} splits")

In [ ]:
plt.figure(figsize=(8, 3.2))
plt.axhline(0, color="grey", linewidth=1)
plt.plot(list(seeds), differences, "o", color="#8C1D2C", label="as the result card ran it")
plt.plot(list(seeds), differences_tuned, "s", color="#26547C", label="logistic regression tuned")
plt.xlabel("seed")
plt.ylabel("difference (percentage points)")
plt.title("What changes when both sides get the same effort")
plt.legend()
plt.tight_layout()
plt.show()

**Answer here.** What happened to gradient boosting's advantage, and why?
Which of the four questions from today does this belong to?

*Your answer:*

## Step B5 — Write the claim

Here is the sentence on the result card:

> Gradient boosting improves accuracy by 3.5 percentage points over logistic
> regression on the Wisconsin breast-cancer data.

**1. Write a claim your own results support.** Say what was compared, on
what data, over how many splits, under what conditions, and how much the
result varied.

*Your claim:*

**2. Name one thing your claim still does not establish.** Use the four
questions — measure, compare, scope, quantify — to find it.

*Your limitation:*

## Before you submit

- Restart the kernel and run every cell from top to bottom.
- Check that every **Answer here** cell is filled in.
- If you used an AI assistant, add a short note saying what it produced and
  what you changed.
- Submit through Moodle.